In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.stats import binomtest

def analyze_patterns(data, min_count=1):
    """Analyze specific patterns in languages without filtering for head-marking."""
    # Use the full dataset without filtering
    full_dataset = data.copy()
    
    total = len(full_dataset)
    print(f"Total languages: {total}")
    
    # Define patterns to analyze
    patterns = [
        ('2', '2', '2'),  # VS, PSSD-PSSR, NAdj
        ('2', '2', '1'),  # VS, PSSD-PSSR, AdjN
        ('1', '2', '2'),  # SV, PSSD-PSSR, NAdj
        ('1', '2', '1'),  # SV, PSSD-PSSR, AdjN
        ('1', '1', '1'),  # SV, PSSR-PSSD, AdjN
        ('1', '1', '2'),  # SV, PSSR-PSSD, NAdj
        ('2', '1', '1'),  # VS, PSSR-PSSD, AdjN
        ('2', '1', '2'),  # VS, PSSR-PSSD, NAdj
    ]
    
    # Feature definitions 
    features = ['GB130', 'GB065', 'GB193']
    value_meanings = {
        'GB130': {'1': 'SV', '2': 'VS', '3': 'Other'},
        'GB065': {'1': 'PSSR-PSSD', '2': 'PSSD-PSSR', '3': 'Other'},
        'GB193': {'1': 'AdjN', '2': 'NAdj', '3': 'Other'}
    }
    
    # Calculate feature distributions (count features in all languages and percentages)
    feature_dists = {}
    for feat in features:
        counts = full_dataset[feat].value_counts()
        percentages = (counts / total * 100).round(1)
        feature_dists[feat] = {
            'counts': counts,
            'percentages': percentages
        }
    
     # Analyze each pattern

    results = []
    for combo in patterns:
        mask = (full_dataset['GB130'] == combo[0]) & \
               (full_dataset['GB065'] == combo[1]) & \
               (full_dataset['GB193'] == combo[2])
        actual = sum(mask)
        
        expected = (total * 
                   (feature_dists['GB130']['counts'].get(combo[0], 0) / total) *
                   (feature_dists['GB065']['counts'].get(combo[1], 0) / total) *
                   (feature_dists['GB193']['counts'].get(combo[2], 0) / total))
        
        ratio = actual / expected if expected > 0 else 0
        
        prob = ((feature_dists['GB130']['counts'].get(combo[0], 0) / total) * #prob features occuring independently
                (feature_dists['GB065']['counts'].get(combo[1], 0) / total) *
                (feature_dists['GB193']['counts'].get(combo[2], 0) / total))
        binom_result = binomtest(actual, n=total, p=prob)
        
        combo_name = '_'.join([
            value_meanings['GB130'].get(combo[0], combo[0]), #returns the corresponding names (PSSR, PSSD etc.)
            value_meanings['GB065'].get(combo[1], combo[1]),
            value_meanings['GB193'].get(combo[2], combo[2])
        ])
        
        status = "insufficient data" if actual < min_count else ( #meaning of categories of represenation
            "significantly over-represented" if binom_result.pvalue < 0.05 and ratio > 1 else
            "significantly under-represented" if binom_result.pvalue < 0.05 else
            "not significant"
        )
        
        results.append({
            'combination': combo_name, #the pattern
            'actual': actual, #how many times the pattern appears (how many languages have it)
            'total': total, #all languages
            'prop': round(actual/total, 3),  #ratio actual to total
            'expected': round(expected, 2), #expected number if features were independent (based on feature frequencies)
            'ratio': round(ratio, 2), #actual ratio
            'binomial_p': binom_result.pvalue, #binomial p-value result
            'status': status #meaning (overrepresented etc.)
        })
    
    return total, feature_dists, results

def create_stratified_sample(data, langs_per_area=20): #sampling anna style
    """Create a stratified sample with fixed number of languages per macroarea."""
    sampled_data = pd.DataFrame()
    macroareas = ["Africa", "Eurasia", "Papunesia", "Australia", "South America", "North America"]
    
    for area in macroareas:
        area_data = data[data['Macroarea'] == area].copy()
        if area_data.empty:
            continue
            
        families = area_data['Family'].unique()
        num_to_sample = min(len(families), langs_per_area)  #smaller of available families and desired 
        selected_families = np.random.choice(families, size=num_to_sample, replace=False) #random selection, no replacement
        
        area_sample = pd.DataFrame()
        for family in selected_families:
            family_data = area_data[area_data['Family'] == family]
            if not family_data.empty:
                sampled_lang = family_data.sample(n=1)
                area_sample = pd.concat([area_sample, sampled_lang])
        
        sampled_data = pd.concat([sampled_data, area_sample])
    
    return sampled_data

def generate_html_report(total, feature_dists, full_results, strat_results):
    """Generate HTML report with analysis results."""
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>Pattern Analysis in Languages</title>
        <style>
            body {{ font-family: Arial, sans-serif; line-height: 1.6; padding: 20px; }}
            .section {{ margin: 20px 0; padding: 20px; background-color: #f8f9fa; border-radius: 8px; }}
            table {{ border-collapse: collapse; width: 100%; margin: 10px 0; }}
            th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
            th {{ background-color: #f5f5f5; }}
            .over {{ color: #2ecc71; font-weight: bold; }}
            .under {{ color: #e74c3c; font-weight: bold; }}
            .expected {{ color: #3498db; }}
            .insufficient {{ color: #95a5a6; }}
        </style>
    </head>
    <body>
        <h1>Pattern Analysis Results</h1>
        
        <div class="section">
            <h2>Feature Distributions</h2>
            <p>Total languages: {total}</p>
    """
    
    for feat, dist in feature_dists.items():
        html += f"""
            <div class="analysis">
                <h3>{feat}</h3>
                <table>
                    <tr><th>Value</th><th>Count</th><th>Percentage</th></tr>
        """
        for val in dist['counts'].index:
            html += f"""
                    <tr>
                        <td>{val}</td>
                        <td>{dist['counts'][val]}</td>
                        <td>{dist['percentages'][val]:.1f}%</td>
                    </tr>
            """
        html += "</table></div>"
    
    html += """
        <div class="section">
            <h2>Pattern Analysis Results</h2>
            <table>
                <tr>
                    <th>Pattern</th>
                    <th>Actual</th>
                    <th>Expected</th>
                    <th>Ratio</th>
                    <th>Status</th>
                </tr>
    """
    
    for row in full_results:
        status_class = 'over' if 'over' in row['status'] else 'under' if 'under' in row['status'] else 'insufficient' if 'insufficient' in row['status'] else 'expected'
        html += f"""
                <tr>
                    <td class="{status_class}">{row['combination']}</td>
                    <td>{row['actual']}</td>
                    <td>{row['expected']}</td>
                    <td>{row['ratio']}</td>
                    <td class="{status_class}">{row['status']}</td>
                </tr>
        """
    
    html += """
            </table>
        </div>
        
        <div class="section">
            <h2>Stratified Sample Results</h2>
            <table>
                <tr>
                    <th>Pattern</th>
                    <th>Mean Count</th>
                    <th>Mean Ratio</th>
                    <th>Std Dev Ratio</th>
                    <th>CI Lower</th>
                    <th>CI Upper</th>
                    <th>Status</th>
                </tr>
    """
    
    for pattern, stats in strat_results.items():
        ratios = np.array(stats['ratios'])
        mean_count = np.mean(stats['counts'])
        mean_ratio = np.mean(ratios)
        std_ratio = np.std(ratios)
        
        # Calculate confidence intervals
        sem = std_ratio / np.sqrt(len(ratios))
        ci_lower = mean_ratio - 1.96 * sem
        ci_upper = mean_ratio + 1.96 * sem
        
        if mean_count >= 3:
            status = "significantly over-represented" if ci_lower > 1 else \
                    "significantly under-represented" if ci_upper < 1 else \
                    "not significant"
        else:
            status = "insufficient data"
            
        status_class = 'over' if 'over' in status else 'under' if 'under' in status else 'insufficient' if 'insufficient' in status else 'expected'
        
        html += f"""
                <tr>
                    <td class="{status_class}">{pattern}</td>
                    <td>{mean_count:.1f}</td>
                    <td>{mean_ratio:.2f}</td>
                    <td>{std_ratio:.2f}</td>
                    <td>{ci_lower:.2f}</td>
                    <td>{ci_upper:.2f}</td>
                    <td class="{status_class}">{status}</td>
                </tr>
        """
    
    html += """
            </table>
        </div>
    </body>
    </html>
    """
    return html

def main():
    # Load data
    print("Loading data...")
    grambank = pd.read_csv('grambank_sane_format.csv')
    languages = pd.read_csv('languages1.csv')
    
    # Add metadata
    metadata = {}
    for _, row in languages.iterrows():
        if pd.notna(row['Name']):
            metadata[row['Name']] = {
                'macroarea': row['Macroarea'],
                'family': row['Family_name']
            }
    
    grambank['Macroarea'] = grambank['Language'].map(lambda x: metadata.get(x, {}).get('macroarea'))
    grambank['Family'] = grambank['Language'].map(lambda x: metadata.get(x, {}).get('family'))
    
    # Convert columns to string
    for col in ['GB431', 'GB433', 'GB130', 'GB065', 'GB193']:
        grambank[col] = grambank[col].astype(str)
    
    # Analyze full dataset
    print("\nAnalyzing full dataset...")
    total, feature_dists, full_results = analyze_patterns(grambank)
    
    # Run stratified analysis
    print("\nRunning stratified analysis...")
    n_samples = 300
    strat_results = defaultdict(lambda: {'ratios': [], 'counts': [], 'props': []})
    
    for i in range(n_samples):
        if i % 10 == 0:
            print(f"Sample {i}/{n_samples}")
        
        sample = create_stratified_sample(grambank, langs_per_area=20)
        _, _, sample_results = analyze_patterns(sample)
        
        for row in sample_results:
            pattern = row['combination']
            entry = strat_results[pattern]
            entry['counts'].append(row['actual'])
            entry['props'].append(row['prop'])
            entry['ratios'].append(row['ratio'])
    
    # Generate and save report
    print("\nGenerating HTML report...")
    html_output = generate_html_report(total, feature_dists, full_results, strat_results)
    
    with open('pattern_analysis_all_languages.html', 'w', encoding='utf-8') as f:
        f.write(html_output)
    
    print("Analysis complete! Results saved to 'pattern_analysis_all_languages_new.html'")

if __name__ == "__main__":
    main()

Loading data...

Analyzing full dataset...
Total languages: 2467

Running stratified analysis...
Sample 0/300
Total languages: 117
Total languages: 116
Total languages: 116
Total languages: 116
Total languages: 115
Total languages: 118
Total languages: 118
Total languages: 117
Total languages: 117
Total languages: 119
Sample 10/300
Total languages: 117
Total languages: 116
Total languages: 116
Total languages: 116
Total languages: 117
Total languages: 118
Total languages: 116
Total languages: 117
Total languages: 117
Total languages: 116
Sample 20/300
Total languages: 116
Total languages: 116
Total languages: 116
Total languages: 116
Total languages: 117
Total languages: 117
Total languages: 115
Total languages: 117
Total languages: 115
Total languages: 118
Sample 30/300
Total languages: 116
Total languages: 116
Total languages: 116
Total languages: 116
Total languages: 116
Total languages: 116
Total languages: 115
Total languages: 117
Total languages: 117
Total languages: 115
Sample 4

In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.stats import binomtest
from tqdm import tqdm

def analyze_patterns(data, min_count=1):
    """Analyze specific patterns in HEAD-MARKING languages."""
    
    # FILTER FOR HEAD-MARKING LANGUAGES
    if 'GB431' not in data.columns or 'GB433' not in data.columns:
        raise ValueError("Required columns GB431 and/or GB433 not found in the data. "
                        "Cannot identify head-marking languages.")
    
    head_marking = data[(data['GB431'] == '1') | (data['GB433'] == '1')].copy()
    
    total = len(head_marking)
    print(f"Total head-marking languages: {total}")
    
    # Define patterns to analyze
    patterns = [
        ('2', '2', '2'),  # VS, PSSD-PSSR, NAdj
        ('2', '2', '1'),  # VS, PSSD-PSSR, AdjN
        ('1', '2', '2'),  # SV, PSSD-PSSR, NAdj
        ('1', '2', '1'),  # SV, PSSD-PSSR, AdjN
        ('1', '1', '1'),  # SV, PSSR-PSSD, AdjN
        ('1', '1', '2'),  # SV, PSSR-PSSD, NAdj
        ('2', '1', '1'),  # VS, PSSR-PSSD, AdjN
        ('2', '1', '2'),  # VS, PSSR-PSSD, NAdj
    ]
    
    # Feature definitions 
    features = ['GB130', 'GB065', 'GB193']
    value_meanings = {
        'GB130': {'1': 'SV', '2': 'VS', '3': 'Other'},
        'GB065': {'1': 'PSSR-PSSD', '2': 'PSSD-PSSR', '3': 'Other'},
        'GB193': {'1': 'AdjN', '2': 'NAdj', '3': 'Other'}
    }
    
    # Calculate feature distributions in HEAD-MARKING languages
    feature_dists = {}
    for feat in features:
        counts = head_marking[feat].value_counts()
        percentages = (counts / total * 100).round(1)
        feature_dists[feat] = {
            'counts': counts,
            'percentages': percentages
        }
    
    # Analyze each pattern
    results = []
    for combo in patterns:
        mask = (head_marking['GB130'] == combo[0]) & \
               (head_marking['GB065'] == combo[1]) & \
               (head_marking['GB193'] == combo[2])
        actual = sum(mask)
        
        # Calculate expected count if features were INDEPENDENT
        expected = (total * 
                   (feature_dists['GB130']['counts'].get(combo[0], 0) / total) *
                   (feature_dists['GB065']['counts'].get(combo[1], 0) / total) *
                   (feature_dists['GB193']['counts'].get(combo[2], 0) / total))
        
        ratio = actual / expected if expected > 0 else 0
        
        # Probability if features occur independently
        prob = ((feature_dists['GB130']['counts'].get(combo[0], 0) / total) *
                (feature_dists['GB065']['counts'].get(combo[1], 0) / total) *
                (feature_dists['GB193']['counts'].get(combo[2], 0) / total))
        
        binom_result = binomtest(actual, n=total, p=prob)
        
        combo_name = '_'.join([
            value_meanings['GB130'].get(combo[0], combo[0]),
            value_meanings['GB065'].get(combo[1], combo[1]),
            value_meanings['GB193'].get(combo[2], combo[2])
        ])
        
        status = "insufficient data" if actual < min_count else (
            "significantly over-represented" if binom_result.pvalue < 0.05 and ratio > 1 else
            "significantly under-represented" if binom_result.pvalue < 0.05 and ratio < 1 else
            "not significant"
        )
        
        results.append({
            'combination': combo_name,
            'actual': actual,
            'total': total,
            'prop': round(actual/total, 3),
            'expected': round(expected, 2),
            'ratio': round(ratio, 2),
            'binomial_p': binom_result.pvalue,
            'status': status
        })
    
    return total, feature_dists, results


def create_stratified_sample(data, langs_per_area=20):
    """Create a stratified sample with fixed number of languages per macroarea."""
    sampled_data = pd.DataFrame()
    
    # Get macroareas from data
    macroareas = data['Macroarea'].dropna().unique()
    
    for area in macroareas:
        area_data = data[data['Macroarea'] == area].copy()
        if area_data.empty:
            continue
            
        families = area_data['Family'].unique()
        families = families[~pd.isna(families)]
        
        if len(families) == 0:
            continue
        
        num_to_sample = min(len(families), langs_per_area)
        selected_families = np.random.choice(families, size=num_to_sample, replace=False)
        
        area_sample = pd.DataFrame()
        for family in selected_families:
            family_data = area_data[area_data['Family'] == family]
            if not family_data.empty:
                sampled_lang = family_data.sample(n=1)
                area_sample = pd.concat([area_sample, sampled_lang])
        
        sampled_data = pd.concat([sampled_data, area_sample])
    
    return sampled_data


def generate_html_report(total, feature_dists, full_results, strat_results):
    """Generate HTML report with analysis results."""
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>Pattern Independence Analysis - Head-Marking Languages</title>
        <style>
            body {{ font-family: Arial, sans-serif; line-height: 1.6; padding: 20px; }}
            .section {{ margin: 20px 0; padding: 20px; background-color: #f8f9fa; border-radius: 8px; }}
            table {{ border-collapse: collapse; width: 100%; margin: 10px 0; }}
            th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
            th {{ background-color: #f5f5f5; }}
            .over {{ color: #2ecc71; font-weight: bold; }}
            .under {{ color: #e74c3c; font-weight: bold; }}
            .expected {{ color: #3498db; }}
            .insufficient {{ color: #95a5a6; }}
            .highlight {{ background-color: #fff3cd; }}
        </style>
    </head>
    <body>
        <h1>Pattern Independence Analysis - Head-Marking Languages</h1>
        <p><strong>Hypothesis:</strong> Tests whether GB130, GB065, and GB193 are independent or correlated in head-marking languages.</p>
        
        <div class="section">
            <h2>Feature Distributions (Head-Marking Languages Only)</h2>
            <p>Total head-marking languages: {total}</p>
    """
    
    for feat, dist in feature_dists.items():
        html += f"""
            <div class="analysis">
                <h3>{feat}</h3>
                <table>
                    <tr><th>Value</th><th>Count</th><th>Percentage</th></tr>
        """
        for val in dist['counts'].index:
            html += f"""
                    <tr>
                        <td>{val}</td>
                        <td>{dist['counts'][val]}</td>
                        <td>{dist['percentages'][val]:.1f}%</td>
                    </tr>
            """
        html += "</table></div>"
    
    html += """
        <div class="section">
            <h2>Full Dataset Pattern Analysis</h2>
            <p><em>Testing: Are actual counts significantly different from expected (if features were independent)?</em></p>
            <table>
                <tr>
                    <th>Pattern</th>
                    <th>Actual</th>
                    <th>Expected</th>
                    <th>Ratio</th>
                    <th>P-value</th>
                    <th>Status</th>
                </tr>
    """
    
    for row in full_results:
        status_class = 'over' if 'over' in row['status'] else 'under' if 'under' in row['status'] else 'insufficient' if 'insufficient' in row['status'] else 'expected'
        html += f"""
                <tr class="{status_class}">
                    <td>{row['combination']}</td>
                    <td>{row['actual']}</td>
                    <td>{row['expected']}</td>
                    <td>{row['ratio']}</td>
                    <td>{row['binomial_p']:.4f}</td>
                    <td>{row['status']}</td>
                </tr>
        """
    
    html += """
            </table>
        </div>
        
        <div class="section">
            <h2>Stratified Sample Results (300 iterations)</h2>
            <p><em>Mean results across genealogically balanced samples</em></p>
            <table>
                <tr>
                    <th>Pattern</th>
                    <th>Mean Count</th>
                    <th>Mean Ratio</th>
                    <th>Std Dev Ratio</th>
                    <th>CI Lower</th>
                    <th>CI Upper</th>
                    <th>Status</th>
                </tr>
    """
    
    for pattern, stats in strat_results.items():
        ratios = np.array(stats['ratios'])
        mean_count = np.mean(stats['counts'])
        mean_ratio = np.mean(ratios)
        std_ratio = np.std(ratios)
        
        # Calculate confidence intervals
        sem = std_ratio / np.sqrt(len(ratios))
        ci_lower = mean_ratio - 1.96 * sem
        ci_upper = mean_ratio + 1.96 * sem
        
        if mean_count >= 3:
            status = "significantly over-represented" if ci_lower > 1 else \
                    "significantly under-represented" if ci_upper < 1 else \
                    "not significant"
        else:
            status = "insufficient data"
            
        status_class = 'over' if 'over' in status else 'under' if 'under' in status else 'insufficient' if 'insufficient' in status else 'expected'
        
        html += f"""
                <tr class="{status_class}">
                    <td>{pattern}</td>
                    <td>{mean_count:.1f}</td>
                    <td>{mean_ratio:.2f}</td>
                    <td>{std_ratio:.2f}</td>
                    <td>{ci_lower:.2f}</td>
                    <td>{ci_upper:.2f}</td>
                    <td>{status}</td>
                </tr>
        """
    
    html += """
            </table>
        </div>
        
        <div class="section">
            <h2>Interpretation Guide</h2>
            <ul>
                <li><strong>Ratio > 1 (Over-represented):</strong> Pattern occurs MORE than if features were independent → features are positively correlated for this combination</li>
                <li><strong>Ratio < 1 (Under-represented):</strong> Pattern occurs LESS than if features were independent → features are negatively correlated for this combination</li>
                <li><strong>Ratio ≈ 1 (Not significant):</strong> Pattern occurs at expected frequency → features appear independent for this combination</li>
                <li><strong>CI excludes 1:</strong> Result is statistically significant across genealogically balanced samples</li>
            </ul>
        </div>
    </body>
    </html>
    """
    return html


def main():
    # Set random seed for reproducibility
    np.random.seed(42)
    
    # Load data
    print("Loading data...")
    try:
        grambank = pd.read_csv('grambank_sane_format.csv')
        languages = pd.read_csv('languages1.csv')
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Please ensure 'grambank_sane_format.csv' and 'languages1.csv' are in the current directory.")
        return
    
    # Add metadata
    metadata = {}
    for _, row in languages.iterrows():
        if pd.notna(row['Name']):
            metadata[row['Name']] = {
                'macroarea': row['Macroarea'],
                'family': row['Family_name']
            }
    
    grambank['Macroarea'] = grambank['Language'].map(lambda x: metadata.get(x, {}).get('macroarea'))
    grambank['Family'] = grambank['Language'].map(lambda x: metadata.get(x, {}).get('family'))
    
    # Verify required columns exist
    required_columns = ['GB431', 'GB433', 'GB130', 'GB065', 'GB193']
    missing_columns = [col for col in required_columns if col not in grambank.columns]
    
    if missing_columns:
        print(f"Error: Required columns missing from dataset: {', '.join(missing_columns)}")
        print(f"\nAvailable columns: {', '.join(grambank.columns.tolist())}")
        return
    
    # Convert columns to string
    for col in required_columns:
        grambank[col] = grambank[col].astype(str)
    
    print(f"Data loaded successfully. Total languages: {len(grambank)}")
    
    # Analyze full dataset
    print("\n" + "="*70)
    print("ANALYZING FULL HEAD-MARKING DATASET")
    print("="*70)
    total, feature_dists, full_results = analyze_patterns(grambank)
    
    # Display full dataset results
    print("\nFull Dataset Results:")
    print("-" * 90)
    print(f"{'Pattern':<30} {'Actual':>8} {'Expected':>8} {'Ratio':>8} {'P-value':>10} {'Status':<25}")
    print("-" * 90)
    for row in full_results:
        status_symbol = "↑" if "over" in row['status'] else "↓" if "under" in row['status'] else "≈"
        print(f"{row['combination']:<30} {row['actual']:>8} {row['expected']:>8.1f} {row['ratio']:>8.2f} {row['binomial_p']:>10.4f} {status_symbol} {row['status']:<25}")
    
    # Run stratified analysis
    print("\n" + "="*70)
    print("RUNNING STRATIFIED ANALYSIS (300 samples)")
    print("="*70)
    n_samples = 300
    strat_results = defaultdict(lambda: {'ratios': [], 'counts': [], 'props': []})
    
    for i in tqdm(range(n_samples), desc="Processing samples"):
        sample = create_stratified_sample(grambank, langs_per_area=20)
        _, _, sample_results = analyze_patterns(sample)
        
        for row in sample_results:
            pattern = row['combination']
            entry = strat_results[pattern]
            entry['counts'].append(row['actual'])
            entry['props'].append(row['prop'])
            entry['ratios'].append(row['ratio'])
    
    # Display stratified results summary
    print("\nStratified Sample Results Summary:")
    print("-" * 110)
    print(f"{'Pattern':<30} {'Mean Count':>11} {'Mean Ratio':>11} {'SD Ratio':>9} {'CI Lower':>9} {'CI Upper':>9} {'Status':<20}")
    print("-" * 110)
    
    for pattern in sorted(strat_results.keys()):
        stats = strat_results[pattern]
        ratios = np.array(stats['ratios'])
        mean_count = np.mean(stats['counts'])
        mean_ratio = np.mean(ratios)
        std_ratio = np.std(ratios)
        
        sem = std_ratio / np.sqrt(len(ratios))
        ci_lower = mean_ratio - 1.96 * sem
        ci_upper = mean_ratio + 1.96 * sem
        
        if mean_count >= 3:
            status = "over-represented" if ci_lower > 1 else \
                    "under-represented" if ci_upper < 1 else \
                    "not significant"
        else:
            status = "insufficient data"
        
        status_symbol = "↑" if "over" in status else "↓" if "under" in status else "≈"
        print(f"{pattern:<30} {mean_count:>11.1f} {mean_ratio:>11.2f} {std_ratio:>9.2f} {ci_lower:>9.2f} {ci_upper:>9.2f} {status_symbol} {status:<20}")
    
    # Generate and save report
    print("\n" + "="*70)
    print("GENERATING HTML REPORT")
    print("="*70)
    html_output = generate_html_report(total, feature_dists, full_results, strat_results)
    
    with open('pattern_independence_head_marking.html', 'w', encoding='utf-8') as f:
        f.write(html_output)
    
    print(f"✓ HTML report saved to: pattern_independence_head_marking.html")
    
    # Save detailed results to CSV
    full_results_df = pd.DataFrame(full_results)
    full_results_df.to_csv('pattern_independence_head_marking_full.csv', index=False)
    print(f"✓ Full results saved to: pattern_independence_head_marking_full.csv")
    
    # Save stratified summary
    strat_summary = []
    for pattern, stats in strat_results.items():
        ratios = np.array(stats['ratios'])
        strat_summary.append({
            'pattern': pattern,
            'mean_count': np.mean(stats['counts']),
            'mean_ratio': np.mean(ratios),
            'std_ratio': np.std(ratios),
            'min_ratio': np.min(ratios),
            'max_ratio': np.max(ratios)
        })
    
    strat_df = pd.DataFrame(strat_summary)
    strat_df.to_csv('pattern_independence_head_marking_stratified.csv', index=False)
    print(f"✓ Stratified summary saved to: pattern_independence_head_marking_stratified.csv")
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE!")
    print("="*70)


if __name__ == "__main__":
    main()

Loading data...
Data loaded successfully. Total languages: 2467

ANALYZING FULL HEAD-MARKING DATASET
Total head-marking languages: 934

Full Dataset Results:
------------------------------------------------------------------------------------------
Pattern                          Actual Expected    Ratio    P-value Status                   
------------------------------------------------------------------------------------------
VS_PSSD-PSSR_NAdj                    56     32.7     1.71     0.0002 ↑ significantly over-represented
VS_PSSD-PSSR_AdjN                    34     15.0     2.26     0.0000 ↑ significantly over-represented
SV_PSSD-PSSR_NAdj                   126    116.4     1.08     0.3465 ≈ not significant          
SV_PSSD-PSSR_AdjN                    22     53.5     0.41     0.0000 ↓ significantly under-represented
SV_PSSR-PSSD_AdjN                    89     62.9     1.41     0.0013 ↑ significantly over-represented
SV_PSSR-PSSD_NAdj                   162    137.0     1.18  

Processing samples:   1%|▏                      | 3/300 [00:00<00:10, 29.42it/s]

Total head-marking languages: 38
Total head-marking languages: 44
Total head-marking languages: 52


Processing samples:   2%|▌                      | 7/300 [00:00<00:09, 30.86it/s]

Total head-marking languages: 44
Total head-marking languages: 44
Total head-marking languages: 50
Total head-marking languages: 43
Total head-marking languages: 48
Total head-marking languages: 47
Total head-marking languages: 49


...

Total head-marking languages: 51
Total head-marking languages: 49
Total head-marking languages: 36
Total head-marking languages: 46
Total head-marking languages: 48
Total head-marking languages: 38
Total head-marking languages: 39


Processing samples:  99%|████████████████████▋| 296/300 [00:09<00:00, 30.21it/s]

Total head-marking languages: 48
Total head-marking languages: 49
Total head-marking languages: 50
Total head-marking languages: 40
Total head-marking languages: 41
Total head-marking languages: 42
Total head-marking languages: 39


Processing samples: 100%|█████████████████████| 300/300 [00:09<00:00, 30.20it/s]


Stratified Sample Results Summary:
--------------------------------------------------------------------------------------------------------------
Pattern                         Mean Count  Mean Ratio  SD Ratio  CI Lower  CI Upper Status              
--------------------------------------------------------------------------------------------------------------
SV_PSSD-PSSR_AdjN                      0.4        0.23      0.33      0.20      0.27 ≈ insufficient data   
SV_PSSD-PSSR_NAdj                      2.5        1.23      0.54      1.16      1.29 ≈ insufficient data   
SV_PSSR-PSSD_AdjN                      6.6        1.53      0.31      1.50      1.56 ↑ over-represented    
SV_PSSR-PSSD_NAdj                      6.3        1.35      0.31      1.32      1.39 ↑ over-represented    
VS_PSSD-PSSR_AdjN                      2.3        3.98      1.97      3.76      4.20 ≈ insufficient data   
VS_PSSD-PSSR_NAdj                      1.5        2.43      1.69      2.24      2.62 ≈ insuffici